# PCA Hyperspectral Analysis — Indian Pines
Uses the same `src/` functions as the Streamlit app and CLI. All numbers are computed from the loaded cube.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from src.data_loader import load_hyperspectral_data, band_statistics
from src.preprocessing import flatten_cube, to_working_float
from src.pca_engine import fit_pca, k_for_variance
from src.reconstruction import reconstruct_cube
from src.metrics import full_report
from src.visualization import false_color, pixel_rmse_map

ds = load_hyperspectral_data('../data/raw')
print(ds.summary())
cube = to_working_float(ds.cube)
H, W, B = cube.shape
print('H,W,B =', H, W, B)

In [ ]:
plt.imshow(false_color(cube)); plt.title('Original false-color'); plt.axis('off'); plt.show()
plt.imshow(cube[:, :, B//2], cmap='gray'); plt.title(f'Band {B//2+1}'); plt.axis('off'); plt.show()
print(band_statistics(cube, B//2))

In [ ]:
X = flatten_cube(cube)
print('X:', X.shape, ' (N=H*W, B=bands)')
eng = fit_pca(X, mode='educational')
evr = eng.result.eigvals_ / eng.result.eigvals_.sum()
print('Top-5 EVR:', evr[:5])
plt.bar(range(1, B+1), evr*100); plt.xlabel('PC'); plt.ylabel('EVR %'); plt.title('Scree'); plt.show()
plt.plot(range(1, B+1), np.cumsum(evr)*100, marker='o', ms=2); plt.ylim(0,100.5); plt.title('Cumulative EVR'); plt.show()

In [ ]:
for target in [0.80, 0.90, 0.95, 0.99]:
    print(target, '-> K =', k_for_variance(eng.result.eigvals_, target))

In [ ]:
K = 20
Z = eng.transform(X, K)
print('Z (compressed):', Z.shape)
recon = reconstruct_cube(Z, eng.result.basis_k(K), eng.result.mean_, H, W)
print('recon:', recon.shape)
rep = full_report(cube, recon, eng.result.eigvals_, K, float(cube.max()))
print(rep)

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(12,4))
ax[0].imshow(false_color(cube)); ax[0].set_title('Original'); ax[0].axis('off')
ax[1].imshow(false_color(recon)); ax[1].set_title('Reconstructed K=20'); ax[1].axis('off')
im = ax[2].imshow(pixel_rmse_map(cube, recon), cmap='hot'); ax[2].set_title('RMSE map'); ax[2].axis('off')
plt.colorbar(im); plt.show()

In [ ]:
print('K | EVR | MSE | PSNR')
for k in [5, 10, 20, 30, 50]:
    Zk = eng.transform(X, k)
    rk = reconstruct_cube(Zk, eng.result.basis_k(k), eng.result.mean_, H, W)
    r = full_report(cube, rk, eng.result.eigvals_, k, float(cube.max()))
    print(f"{k:3d} | {r['explained_variance']*100:6.2f}% | {r['mse']:.1f} | {r['psnr_db']:.2f} dB")